# Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Базовые модели
Logistic Regression, Random Forest, CatBoost

In [2]:
df = pd.read_csv('../data/heart.csv')

X = df.drop(columns='HeartDisease')
y = df['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# медианы только по train
medians = {col: X_train[col].replace(0, np.nan).median()
           for col in ['Cholesterol', 'RestingBP']}

def clean(part):
    part = part.copy()
    for col in ['Cholesterol', 'RestingBP']:
        part[col] = part[col].replace(0, np.nan).fillna(medians[col])
    part['Oldpeak'] = part['Oldpeak'].clip(lower=0)
    return part

def encode(part):
    part = part.copy()
    part['Sex'] = (part['Sex'] == 'M').astype(int)
    part['ExerciseAngina'] = (part['ExerciseAngina'] == 'Y').astype(int)
    part['ST_Slope'] = part['ST_Slope'].map({'Up': 0, 'Flat': 1, 'Down': 2})
    return pd.get_dummies(part, columns=['ChestPainType', 'RestingECG'],
                          drop_first=True, dtype=int)

X_train = encode(clean(X_train))
X_test = encode(clean(X_test))
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

num_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(X_train.shape, X_test.shape)

(734, 14) (184, 14)


In [3]:
models = {
    'LogReg': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=200, depth=4, learning_rate=0.1,
                                   random_seed=42, verbose=False),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, proba),
    })

test_df = pd.DataFrame(rows).round(3)
test_df

,model,accuracy,precision,recall,f1,roc_auc
0,LogReg,0.870,0.861,0.912,0.886,0.907
1,RandomForest,0.870,0.890,0.873,0.881,0.933
2,CatBoost,0.864,0.889,0.863,0.876,0.920


### Выводы по базовым моделям на тесте

1. Все три модели дают 86–87% accuracy против 55.3%
   у константного прогноза большинства.
2. **LogReg:** лучший recall (0.912) и F1 (0.886) - пропускает меньше всего
   больных, но чаще ложно тревожится (precision 0.861) и хуже всех ранжирует
   риск (ROC-AUC 0.907).
3. **CatBoost:** на дефолтных параметрах стабильно чуть ниже RF по всем
   метрикам - преимуществ не дал.
4. **RandomForest:** лучший ROC-AUC (0.933) и precision (0.890) при том же
   accuracy (0.870). Отставание по F1 всего 0.005 (~1 объект из 184) -
   статистически незначимо.

   **RandomForest** - лучшая различающая способность и
   точность положительных прогнозов. recall (0.873).

In [4]:
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=5, scoring=scoring)
    cv_rows.append({'model': name,
                    **{m: scores[f'test_{m}'].mean() for m in scoring}})

cv_df = pd.DataFrame(cv_rows).round(3)
cv_df

,model,accuracy,precision,recall,f1,roc_auc
0,LogReg,0.845,0.855,0.867,0.861,0.911
1,RandomForest,0.853,0.849,0.894,0.870,0.918
2,CatBoost,0.857,0.864,0.882,0.872,0.923


### Выводы: кросс-валидация и сравнение с тестом

1. **Устойчивость:** разрыв CV - test по всем метрикам <= 0.025 -
   переобучения нет, модели обобщают стабильно.
2. **На CV модели почти равны:** CatBoost формально лидирует по
   accuracy/f1/ROC-AUC (0.857/0.872/0.923), RandomForest - по recall (0.894).
   различия <= 0.005, т.е. в пределах шума.
3. **Тест:** RF лучший по ROC-AUC (0.933) и precision (0.890),
   LogReg - по recall (0.912). CatBoost стабилен посередине.
4. **Явного победителя на дефолтах нет** - выбор финальной модели возможен после
   после подбора гиперпараметров.
5. **Кандидат - RandomForest** - лучший CV-recall (0.894,
   медицински важный) и лучший тестовый ROC-AUC.
   CatBoost - резервный и кандидат.